In [66]:
import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math

from pathlib import Path

from schema_json import ReportData

In [111]:
DEVELOPER_PROMPT_FILE_NAME = "system_prompt_3_openai.txt"

TRAINING_DATA_FILE_NAME = "data_luca_train.jsonl"

In [68]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

MODEL = 'gpt-4o-mini'

In [112]:
#Carica developer prompt da file
developer_prompt_path = Path('../models/prompts/' + DEVELOPER_PROMPT_FILE_NAME)

with open(developer_prompt_path, "r", encoding="utf-8") as f:
    developer_prompt = f.read()

print(developer_prompt)

# Load training data from file
training_data_path = Path('../data/ft-dataset/' + TRAINING_DATA_FILE_NAME)

with open(training_data_path, "r", encoding="utf-8") as f:
    training_data = [json.loads(line) for line in f]

test_report = training_data[0]['messages'][1]['content'] 
print(test_report)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo in uno schema JSON.
RM ADDOME INFERIORE (S/C MDC)
ESAME ESEGUITO MEDIANTE SEQUENZE FRFSE, DWI E DISCO PRIMA E DOPO SOMMINISTRAZIONE IV DI MDC PARAMAGNETICO, SECONDO PIANI MULTIPLI, PREVIA DISTENSIONE DEL RETTO CON GEL E IPOTONIZZANTE INTRAMUSCOLO.
IN CORRISPONDENZA DEL RETTO ALTO E DEL PASSAGGIO RETTO-SIGMA ISPESSIMENTO PROLIFERATIVO DELLA PARETE, INTERESSATA CIRCONFERENZIALMENTE DETERMINANDO  MARCATA RIDUZIONE DI CALIBRO /STENOSI DEL LUME ; LA  LESIONE, CHE PRESENTA MARGINE INFERIORE A CIRCA 11 CM DALLO SFINTERE ANALE ED ESTENSIONE CRANIO-CAUDALE DI CIRCA 5,5 CM, E IPERINTENSA NELLE SEQUENZE T2-DIPENDENTI, PRESENTA  RESTRIZIONE DELLA DIFFUSIONE PROTONICA , CORRISPETTIVA IPOINTENSITA NELLA MAPPA ADC ; DOPO MDC PRESENTA ABNORME IMPREGNAZIONE NELL'ACQUISZIONE PERFUSIONALE
LA PA

In [72]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": developer_prompt},
        {
            "role": "user",
            "content": test_report,
        },
    ],
    text_format=ReportData,
)

report_data = response.output_parsed

In [73]:
print(report_data)

morfologia=<Morfologia.solido_anulare: 'solido_anulare'> posizione=<Posizione.alto: 'alto'> spessore_parietale=None estensione_cranio_caudale=5 distanza_oai=11 riflessione_peritoneale_anteriore=<RiflessionePeritonealeAnteriore.sotto: 'sotto'> infiltrazione_tessuto_adiposo=<InfiltrazioneTessutoAdiposo.si_5mm: 'si_5mm'> infiltrazione_sfinteri=<InfiltrazioneSfinteri.no: 'no'> infiltrazione_organi_extra=<InfiltrazioneOrganiExtra.no: 'no'> coinvolgimento_riflessione_peritoneale=<CoinvolgimentoRiflessionePeritoneale.no: 'no'> coinvolgimento_fascia_mesorettale=<CoinvolgimentoFasciaMesorettale.no: 'no'> linfonodi_sospetti=10 sedi_linfonodi_locoregionali=SediLinfonodiLocoregionali(mesorettali=True, rettali_superiori=False, mesenterici_inferiori=False, iliaci_interni=True, otturatori=True, sacrali=False, inguinali_sotto_dentata=False) sedi_linfonodi_non_locoregionali=SediLinfonodiNonLocoregionali(inguinali=False, iliaci_esterni=True, iliaci_comuni=True, paraortici=False, altri=False) depositi_tu

In [74]:
type(report_data.model_dump())

dict

In [75]:
report_data.model_dump()

{'morfologia': <Morfologia.solido_anulare: 'solido_anulare'>,
 'posizione': <Posizione.alto: 'alto'>,
 'spessore_parietale': None,
 'estensione_cranio_caudale': 5,
 'distanza_oai': 11,
 'riflessione_peritoneale_anteriore': <RiflessionePeritonealeAnteriore.sotto: 'sotto'>,
 'infiltrazione_tessuto_adiposo': <InfiltrazioneTessutoAdiposo.si_5mm: 'si_5mm'>,
 'infiltrazione_sfinteri': <InfiltrazioneSfinteri.no: 'no'>,
 'infiltrazione_organi_extra': <InfiltrazioneOrganiExtra.no: 'no'>,
 'coinvolgimento_riflessione_peritoneale': <CoinvolgimentoRiflessionePeritoneale.no: 'no'>,
 'coinvolgimento_fascia_mesorettale': <CoinvolgimentoFasciaMesorettale.no: 'no'>,
 'linfonodi_sospetti': 10,
 'sedi_linfonodi_locoregionali': {'mesorettali': True,
  'rettali_superiori': False,
  'mesenterici_inferiori': False,
  'iliaci_interni': True,
  'otturatori': True,
  'sacrali': False,
  'inguinali_sotto_dentata': False},
 'sedi_linfonodi_non_locoregionali': {'inguinali': False,
  'iliaci_esterni': True,
  'ilia

In [92]:
print(report_data.morfologia.value)

solido_anulare


In [93]:
report_data.linfonodi_sospetti

10

In [108]:
report_data.sedi_linfonodi_locoregionali.model_dump()

{'mesorettali': True,
 'rettali_superiori': False,
 'mesenterici_inferiori': False,
 'iliaci_interni': True,
 'otturatori': True,
 'sacrali': False,
 'inguinali_sotto_dentata': False}

In [113]:
display(training_data[0]['messages'][2]['content'])

{'morfologia': 'solido_anulare',
 'spessore_parietale': None,
 'estensione_cranio_caudale': None,
 'distanza_oai': None,
 'posizione': 'giunzione',
 'carcinosi_peritoneale': None,
 'lesioni_ossee': None,
 'riflessione_peritoneale_anteriore': 'cavallo',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'no',
 'linfonodi_sospetti': 10.0,
 'sedi_locoregionali': {'mesorettali': True,
  'rettali_superiori': True,
  'mesenterici_inferiori': False,
  'iliaci_interni': True,
  'otturatori': True,
  'sacrali': False,
  'inguinali_sotto_dentata': False},
 'sedi_non_locoregionali': {'inguinali': False,
  'iliaci_esterni': True,
  'iliaci_comuni': True,
  'paraortici': False,
  'altri': False},
 'depositi_tumorali': 'no',
 'numero_depositi': 0.0,
 'emvi_esteso': 'no'}